# Hybrid Production Architecture

## Scenario: select the least autonomous reliable route

Northstar receives a known status lookup, intermittent EU failures, and a 35% enterprise-EU conversion drop. We route each to deterministic workflow, bounded investigator, or specialist team, then pass every result through policy and approval controls.

**Safety boundary:** agents only produce evidence/proposals. Identity, tool authorization, budgets, approval, execution, and rollback are deterministic application controls.

![Production architecture trade-offs](../../../assets/reliable-agentic-systems-tradeoffs.svg)

Hybrid means autonomy is a deliberate route selected from task features and evaluation evidence—not a default model behavior. More autonomy must earn its increased cost, latency, attack surface, and coordination risk.

## 1. Route contract and policy plane

Normalize tenant/identity, task family, known path, ambiguity, impact, tool/data scope, deadline, cost/action budget, and required output. Route deterministically: known low-risk work → workflow; ambiguous bounded evidence → single agent; high-impact/separable evidence → team plus reviewer. Independent policy validates trust/provenance/tenant, tool arguments/resources, rate/budget, evidence/output, authorization, and audit.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'curriculum').exists(): ROOT=next(p for p in (ROOT,*ROOT.parents) if (p/'curriculum').exists())
sys.path.insert(0,str(ROOT/'curriculum'/'advanced'/'05-incident-response-capstone'))
from agentops_lab.hybrid_production_architecture import run_examples,IncidentRequest,plan_architecture
plans=run_examples()
for plan in plans:
    print(plan.route,'->',plan.architecture)
    print(' checks:',', '.join(plan.policy_checks),'approval:',plan.approval_required)
    print(' action:',plan.action)
assert [p.route for p in plans]==['simple_lookup','investigation','high_risk_case']

simple_lookup -> deterministic workflow
 checks: tool allowlist, evidence required, budget limit, audit log approval: False
 action: Read service status and format a report.
investigation -> single bounded agent
 checks: tool allowlist, evidence required, budget limit, audit log, confidence threshold approval: False
 action: Investigate, summarize evidence, and propose a non-destructive recommendation.
high_risk_case -> agent team inside deterministic workflow
 checks: tool allowlist, evidence required, budget limit, audit log, human approval, rollback preview, customer-impact review approval: True
 action: Run specialist team, apply policy checks, pause for human approval before rollback or notification.


## 2. Complex case: evidence, team, approval, recovery

For the enterprise-EU drop, a team may collect attributed metrics/logs, deployment history, customer impact, synthesis, and risk review. It remains a read-only evidence plane. The controller persists the proposal and its action fingerprint, validates tenant/tool/policy/budget/evidence, pauses for fresh human approval, and executes only through idempotent action services. If an artifact is missing, tools fail, budget expires, or reviewer rejects the claim, it escalates or safely stops.

Use LangGraph where durable state/interrupt/recovery are needed; use a compact managed agent SDK for a bounded single loop; use team frameworks only when baseline evaluation proves specialization helps.

In [2]:
# Failure / fallback: a medium-risk but known path stays deterministic.
known=IncidentRequest('format status after cache miss','medium','low',True,'moderate')
p=plan_architecture(known)
print(p)
assert p.architecture=='deterministic workflow'

# High risk requires approval even when the team has a plausible recommendation.
high=plans[-1]
assert high.approval_required and 'human approval' in high.policy_checks

RoutedPlan(route='simple_lookup', architecture='deterministic workflow', reason='The steps are known and the task does not need dynamic tool selection.', policy_checks=['tool allowlist', 'evidence required', 'budget limit', 'audit log'], approval_required=False, action='Read service status and format a report.')


## 3. Evaluation and release gate

Measure routing accuracy, task/evidence correctness, unsafe-action blocks, approval correctness, tool trajectory, cost per successful safe task, p95 latency, fallback/recovery, and tenant/risk slices. Test missing/conflicting evidence, poisoning, tool/model outage, duplicate approval, budget exhaustion, and rollback. Roll out routes incrementally; retain a simpler fallback and an immediate kill/revoke path.

**Exercises:** add a customer-message approval route; choose a model/tool fallback; compare team and single-agent evidence/cost; write a release gate; design durable checkpoint/replay controls.

References: [OpenAI agent guide](https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/), [Anthropic effective agents](https://www.anthropic.com/engineering/building-effective-agents), [NIST AI RMF](https://www.nist.gov/itl/ai-risk-management-framework).